# Modelagem e treino — SOP (PCOS)

Treino de classificadores **apenas no conjunto de treino** — sem avaliar o teste.

- **Entrada:** `src/db/outputs/polycystic-ovary-syndrome-pcos/02-preprocessamento/artifacts/dados_preprocessados.joblib`
- **Artefato:** `src/db/outputs/polycystic-ovary-syndrome-pcos/03-modelagem-treino/artifacts/modelos_treinados.joblib`
- **Próximo:** notebook `04_avaliacao_metricas.ipynb`


## 1 - Setup e carregamento


In [1]:
from pathlib import Path
import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

def encontrar_raiz() -> Path:
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "src" / "db").is_dir():
            return candidato
    raise FileNotFoundError("Raiz do repositório não encontrada.")

RAIZ = encontrar_raiz()
OUTPUT_BASE = RAIZ / "src/db/outputs/polycystic-ovary-syndrome-pcos"

PASTA_ENTRADA = OUTPUT_BASE / "02-preprocessamento"
PASTA_BASE = OUTPUT_BASE / "03-modelagem-treino"
PASTA_ARTEFATOS = PASTA_BASE / "artifacts"
CAMINHO_ARTEFATO_02 = PASTA_ENTRADA / "artifacts" / "dados_preprocessados.joblib"

PASTA_ARTEFATOS.mkdir(parents=True, exist_ok=True)

if not CAMINHO_ARTEFATO_02.exists():
    raise FileNotFoundError("Execute o notebook 02 primeiro.")

pacote = joblib.load(CAMINHO_ARTEFATO_02)
atributos_treino = pacote["atributos_treino"]
atributos_teste = pacote["atributos_teste"]
rotulo_treino = pacote["rotulo_treino"]
rotulo_teste = pacote["rotulo_teste"]
colunas_atributos = pacote["colunas_atributos"]
preprocessador = pacote["preprocessador"]

print(f"Treino: {len(atributos_treino)} | Teste: {len(atributos_teste)} (guardado)")
print(f"Atributos: {len(colunas_atributos)}")


Treino: 432 | Teste: 109 (guardado)
Atributos: 9


## 2 - Treinar pipelines


In [2]:
SEMENTE = 42

def criar_pipeline(classificador):
    return Pipeline([
        ("preprocessador", preprocessador),
        ("classificador", classificador),
    ])

modelos_config = {
    "Regressão Logística": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEMENTE),
    "Árvore de Decisão": DecisionTreeClassifier(class_weight="balanced", random_state=SEMENTE),
    "SVM": CalibratedClassifierCV(
        estimator=SVC(kernel="rbf", class_weight="balanced"),
        method="sigmoid",
        cv=5,
        ensemble=False,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEMENTE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=SEMENTE,
        n_jobs=-1,
    ),
}

modelos_treinados = {}
for nome, clf in modelos_config.items():
    pipeline = criar_pipeline(clf)
    pipeline.fit(atributos_treino, rotulo_treino)
    modelos_treinados[nome] = pipeline
    print(f"{nome}: OK")


Regressão Logística: OK
Árvore de Decisão: OK
SVM: OK


Gradient Boosting: OK


Random Forest: OK


## 3 - Salvar artefato


In [3]:
joblib.dump(
    {
        "modelos_treinados": modelos_treinados,
        "rotulo_teste": rotulo_teste,
        "atributos_teste": atributos_teste,
        "colunas_atributos": colunas_atributos,
    },
    PASTA_ARTEFATOS / "modelos_treinados.joblib",
)
print("Artefato salvo:", PASTA_ARTEFATOS / "modelos_treinados.joblib")


Artefato salvo: E:\_git\fiap\fase1\tech-challenge-fase1\src\db\outputs\polycystic-ovary-syndrome-pcos\03-modelagem-treino\artifacts\modelos_treinados.joblib


> **Não avaliar o teste aqui.** Métricas ficam no notebook 04.


In [4]:
assert len(modelos_treinados) >= 2
assert (PASTA_ARTEFATOS / "modelos_treinados.joblib").exists()
print("Validação OK")


Validação OK
